# 14 — UCI Multicenter Real Data

**Mục tiêu:** hợp nhất bốn cohort thật Cleveland, Hungarian, Switzerland và Long Beach VA; giữ đủ 13 feature; xử lý missing hoàn toàn bên trong Pipeline; đánh giá khả năng tổng quát hóa sang bệnh viện chưa từng xuất hiện trong training.

> Đây là nghiên cứu trên dữ liệu công khai, không phải bằng chứng lâm sàng hay công cụ chẩn đoán.

In [ ]:
!pip install -q lightgbm seaborn

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, brier_score_loss, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)

## 1. Tải bốn cohort processed từ UCI

Mỗi file có 13 feature và `num`. Ký hiệu `?` được đọc thành `NaN`; target binary là `num > 0`. Cột `site` chỉ phục vụ audit và chia external test, không đưa vào model.

In [ ]:
COLUMNS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'
]
FEATURES = COLUMNS[:-1]
NUMERICAL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {
    'cleveland': 'processed.cleveland.data',
    'hungarian': 'processed.hungarian.data',
    'switzerland': 'processed.switzerland.data',
    'va': 'processed.va.data',
}

frames = []
for site, filename in FILES.items():
    url = f'{BASE_URL}/{filename}'
    part = pd.read_csv(url, names=COLUMNS, na_values=['?'], skipinitialspace=True)
    part = part.apply(pd.to_numeric, errors='coerce')
    part['site'] = site
    part['target'] = (part['num'] > 0).astype('int8')
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
print('Combined shape:', df.shape)
display(df.head())

## 1.1. Xuất dataset UCI multicenter mới

Tạo hai file CSV nhưng **không impute trước**:

- `uci_multicenter_raw.csv`: bảo toàn 13 feature, `num`, `target`, `site`.
- `uci_multicenter_train_ready.csv`: 13 feature, `target`, `site`.

Cột `site` được giữ để audit và chia Leave-One-Center-Out, nhưng không được đưa vào model.

In [ ]:
import shutil

OUTPUT_DIR = Path('uci_multicenter_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_COLUMNS = FEATURES + ['num', 'target', 'site']
TRAIN_READY_COLUMNS = FEATURES + ['target', 'site']

raw_path = OUTPUT_DIR / 'uci_multicenter_raw.csv'
train_ready_path = OUTPUT_DIR / 'uci_multicenter_train_ready.csv'

df[RAW_COLUMNS].to_csv(raw_path, index=False)
df[TRAIN_READY_COLUMNS].to_csv(train_ready_path, index=False)

readme_path = OUTPUT_DIR / 'README_DATASET.txt'
readme_path.write_text(
    'UCI Heart Disease Multicenter Dataset\n'
    'Sources: Cleveland, Hungarian, Switzerland, Long Beach VA\n'
    'Features: 13\n'
    'target: 0 = no disease, 1 = disease (original num > 0)\n'
    'site: source cohort; use for audit/splitting, not as a model feature\n'
    'Missing values are intentionally preserved for fold-safe imputation.\n'
    'Source: https://archive.ics.uci.edu/dataset/45/heart+disease\n',
    encoding='utf-8'
)

zip_path = Path(shutil.make_archive('uci_multicenter_data', 'zip', root_dir=OUTPUT_DIR))

print('Raw dataset:', raw_path, df[RAW_COLUMNS].shape)
print('Train-ready dataset:', train_ready_path, df[TRAIN_READY_COLUMNS].shape)
print('ZIP package:', zip_path)
display(df[RAW_COLUMNS].head())

In [ ]:
# Chạy cell này trên Google Colab để tải cả hai CSV và README trong một file ZIP.
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Không chạy trong Colab. File ZIP nằm tại:', zip_path.resolve())

## 2. Kiểm tra chất lượng và khác biệt giữa bệnh viện

Không tự động biến các giá trị `0` thành missing. Trước tiên phải quan sát theo từng site và chỉ sửa sentinel khi có bằng chứng từ data dictionary hoặc chuyên gia lâm sàng.

In [ ]:
site_summary = df.groupby('site').agg(
    rows=('target', 'size'),
    disease_count=('target', 'sum'),
    positive_rate=('target', 'mean'),
    age_mean=('age', 'mean')
)
site_summary['missing_cells'] = df.groupby('site')[FEATURES].apply(lambda x: int(x.isna().sum().sum()))
site_summary['missing_rate'] = site_summary['missing_cells'] / (site_summary['rows'] * len(FEATURES))
display(site_summary.round(4))

missing_by_site = df.groupby('site')[FEATURES].apply(lambda x: x.isna().mean()).T
display((missing_by_site * 100).round(1).style.background_gradient(cmap='Reds', axis=None).format('{:.1f}%'))

print('Duplicate rows excluding site:', int(df[COLUMNS].duplicated().sum()))
print('Target distribution:')
display(pd.crosstab(df['site'], df['target'], margins=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=site_summary.reset_index(), x='site', y='positive_rate', ax=axes[0])
axes[0].set_title('Disease prevalence by site')
axes[0].set_ylim(0, 1)
sns.heatmap(missing_by_site * 100, annot=True, fmt='.0f', cmap='Reds', ax=axes[1])
axes[1].set_title('Missing rate (%) by feature and site')
plt.tight_layout()
plt.show()

## 3. Pipeline giữ đủ feature và chống leakage

- Numerical: median imputation + missing indicator + StandardScaler.
- Categorical: most-frequent imputation + missing indicator + OneHotEncoder.
- Mỗi pipeline được fit lại từ đầu trong từng fold Leave-One-Center-Out.

In [ ]:
def make_preprocessor():
    numeric = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
    ])
    categorical = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ])
    return ColumnTransformer([
        ('numerical', numeric, NUMERICAL_FEATURES),
        ('categorical', categorical, CATEGORICAL_FEATURES),
    ])

def make_models():
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=250, learning_rate=0.03, num_leaves=15,
            min_child_samples=15, subsample=0.9, colsample_bytree=0.9,
            reg_lambda=1.0, class_weight='balanced',
            random_state=RANDOM_STATE, verbosity=-1
        ),
    }

def metric_record(model_name, test_site, y_true, probability, seconds):
    prediction = (probability >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        'model': model_name, 'test_site': test_site, 'test_rows': len(y_true),
        'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability),
        'brier': brier_score_loss(y_true, probability),
        'false_negatives': int(fn), 'fit_seconds': seconds,
    }

## 4. Leave-One-Center-Out external validation

Ở mỗi lượt, một bệnh viện bị khóa hoàn toàn để làm test. Đây là đánh giá khó hơn random split nhưng gần với tình huống triển khai sang cơ sở mới hơn.

In [ ]:
records = []
fitted_by_fold = {}

for test_site in FILES:
    train_df = df[df['site'] != test_site].copy()
    test_df = df[df['site'] == test_site].copy()
    X_train, y_train = train_df[FEATURES], train_df['target']
    X_test, y_test = test_df[FEATURES], test_df['target']

    for model_name, classifier in make_models().items():
        pipeline = Pipeline([
            ('preprocessor', make_preprocessor()),
            ('classifier', classifier),
        ])
        started = time.perf_counter()
        pipeline.fit(X_train, y_train)
        elapsed = time.perf_counter() - started
        probability = pipeline.predict_proba(X_test)[:, 1]
        records.append(metric_record(model_name, test_site, y_test, probability, elapsed))
        fitted_by_fold[(model_name, test_site)] = pipeline

loco_results = pd.DataFrame(records).sort_values(['test_site', 'roc_auc'], ascending=[True, False])
display(loco_results.round(4))

## 5. Tổng hợp: trung bình và worst hospital

Model phù hợp triển khai không chỉ cần điểm trung bình tốt mà còn cần hạn chế suy giảm ở bệnh viện khó nhất.

In [ ]:
model_summary = loco_results.groupby('model').agg(
    roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'),
    roc_auc_worst=('roc_auc', 'min'),
    recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'),
    recall_worst=('recall', 'min'),
    specificity_mean=('specificity', 'mean'),
    f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'),
    false_negatives_total=('false_negatives', 'sum'),
    fit_seconds_mean=('fit_seconds', 'mean'),
).sort_values(['roc_auc_worst', 'recall_worst'], ascending=False)
display(model_summary.round(4))

plot_data = loco_results.pivot(index='test_site', columns='model', values='roc_auc')
plot_data.plot(kind='bar', figsize=(10, 4), ylim=(0.5, 1.0), rot=0)
plt.ylabel('ROC-AUC')
plt.title('External performance on each held-out hospital')
plt.axhline(0.5, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

## 6. Phân tích missingness có ảnh hưởng ra sao

Bảng này ghép missing rate của site test với kết quả model. Đây là phân tích mô tả, chưa chứng minh quan hệ nhân quả.

In [ ]:
site_quality = site_summary[['missing_rate', 'positive_rate']].reset_index().rename(columns={'site': 'test_site'})
impact_table = loco_results.merge(site_quality, on='test_site', how='left')
display(impact_table[[
    'model', 'test_site', 'test_rows', 'missing_rate', 'positive_rate',
    'roc_auc', 'recall', 'specificity', 'brier', 'false_negatives'
]].sort_values(['model', 'missing_rate']).round(4))

## 7. Kết luận tự động và quy tắc quyết định

Ưu tiên `worst-site ROC-AUC`, sau đó `worst-site recall`, Brier và false negatives. Không chọn model chỉ vì thắng trên Cleveland.

In [ ]:
winner = model_summary.reset_index().iloc[0]
print('MULTICENTER ROBUSTNESS CANDIDATE')
display(winner)
print('\nDiễn giải: đây là empirical winner trên bốn cohort UCI theo LOCO, chưa phải model được xác nhận lâm sàng.')
print('Bước tiếp theo: tune threshold trên training hospitals, kiểm tra calibration, rồi xác nhận trên nguồn dữ liệu độc lập mới.')

## Checklist chống kết luận quá mức

- [x] Dùng bệnh nhân thật từ nhiều cohort.
- [x] Giữ đủ 13 feature.
- [x] Imputation chỉ fit trong fold training.
- [x] Không đưa `site` vào model.
- [x] Báo cáo trung bình và bệnh viện tệ nhất.
- [ ] Xác minh ý nghĩa các giá trị 0 bất thường trước khi đổi thành missing.
- [ ] Tune threshold bằng nested validation, không tune trên held-out hospital.
- [ ] External validation trên dữ liệu bệnh viện mới.